# P24 — Storage Optimization / Compression (the SO in SO-DAGHAR)

**The pivot:** performance work is locked (78.10 ± 0.14% DG; DA rung closed). P24
measures the **storage-accuracy trade-off** of the final system, including the
**original P1/P6 SO suite (NC, AW, RED, GS)** applied to the best model.

| Axis | Techniques | Compute |
|---|---|---|
| Architecture capacity | full / compact / tiny retrains | ~3 GNN runs |
| Weights compression | fp16, dynamic int8, pruning 30/50/70 | inference only |
| **Classic SO: edges** | **RED, GS** vs Pearson top-k at matched budgets | **inference only** |
| **Classic SO: input** | **NC** (accel_only/gyro_only/no_gz), **AW** (40/30) | ~5 GNN runs |
| Graph density | Pearson top-k curve from P21 dumps | zero |

> **Final_PROJECT note (updated 2026-09-26).** All paths now come from
> `Final_PROJECT/project_paths.py` via the first code cell below — the only thing a new user may
> need to edit is the DAGHAR data path in that file (and `FINAL_PROJECT_ROOT` below, only if the
> folder isn't found automatically). **Saved cell outputs are from the original thesis run**, when
> this code lived at `/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression`; they are kept as the record of that run.
> All training scripts are **resumable**: re-running this notebook in place skips any fold whose
> result files already exist, so it will not overwrite the thesis results.

## 1. Mount & paths

In [1]:
# ── Mount Drive & load Final_PROJECT paths (single source: Final_PROJECT/project_paths.py) ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob

# EDIT ONLY IF you uploaded Final_PROJECT somewhere else on your Drive
# (if this path doesn't exist, the folder is searched for automatically).
FINAL_PROJECT_ROOT = "/content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT"

if not os.path.isfile(os.path.join(FINAL_PROJECT_ROOT, "project_paths.py")):
    hits = []
    for depth in range(0, 5):
        hits = glob.glob("/content/drive/MyDrive/" + "*/" * depth + "Final_PROJECT/project_paths.py")
        if hits: break
    assert hits, "Final_PROJECT not found on Drive — set FINAL_PROJECT_ROOT above"
    FINAL_PROJECT_ROOT = os.path.dirname(hits[0]); print("auto-detected Final_PROJECT at", FINAL_PROJECT_ROOT)
sys.path.insert(0, FINAL_PROJECT_ROOT)
import project_paths as PP            # the DAGHAR data path is set inside project_paths.py

PROJECT_PATH = PP.activate("p24")   # exports DATA_PATH, checks data, cd's into this component
DATA_PATH    = PP.DATA_PATH
P21_BASE     = PP.P21_BASE_S42     # frozen ConvNet probs (seed 42) for the fused numbers
P21_CFG      = PP.P21_SWEEP_CFG    # P21 Stage-A dumps (edge-density curve source)
assert os.path.isdir(P21_BASE), "missing P21_BASE (frozen ConvNet probs for fused numbers)"
assert os.path.isdir(P21_CFG), "missing P21_CFG (edge-density curve source)"

Mounted at /content/drive
Final_PROJECT paths
  PROJECT_ROOT : /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT
  DATA_PATH    : /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  component    : p24 -> /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT/02_Contribution2_SO_Framework/P24_SO_Compression
  cwd          : /content/drive/MyDrive/Thesis/SO_Experiments/Final_PROJECT/02_Contribution2_SO_Framework/P24_SO_Compression


## 2. Setup

In [2]:
import setup_colab
setup_colab.install_deps(); assert setup_colab.verify(), "fix DATA_PATH"

Installing PyTorch Geometric ...
Done.

Verifying imports ...
  OK torch 2.11.0+cu128  CUDA=True
     GPU Tesla T4
  OK torch_geometric 2.8.0.post1

Checking data at: /content/drive/MyDrive/HAR-Datasets/DAGHAR/standardized_view
  OK KuHar              3 CSVs
  OK MotionSense        3 CSVs
  OK RealWorld-Thigh    3 CSVs
  OK RealWorld-Waist    3 CSVs
  OK UCI                3 CSVs
  OK WISDM              3 CSVs


## 3. Smoke test (~4 min)
One fold, 2 epochs, tiny variant: train → checkpoint → compression eval runs through.

In [3]:
# Smoke test writes to a SEPARATE folder, so the real results/checkpoints/tiny is never touched.
!python -m p24.train_save --data_path "$DATA_PATH" --targets KuHar --epochs 2 --log_every 1 --variant tiny --out_root results/_smoke_checkpoints
import torch; ck = torch.load("results/_smoke_checkpoints/tiny/ckpt_KuHar.pt", map_location="cpu", weights_only=False)
print("smoke OK: ckpt params =", ck["n_params"])
import shutil; shutil.rmtree("results/_smoke_checkpoints"); print("smoke checkpoints removed")

[P24] train+save | arm=tiny {'cnn_embed_dim': 16, 'gnn_hidden': 16, 'cnn_channels': [8, 16]} | nc=None aw=- | seed=42 | targets=['KuHar'] | out=results/_smoke_checkpoints/tiny
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100%, (4, 5):100%, (0, 1):100%, (1, 4):100%
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 18 edges, 

## 4. Train + save the three capacity variants (~1.5 h)
`full` = the locked P21 winner architecture (also gives us its checkpoints);
`compact` = half width; `tiny` = quarter width. Resumable per fold.

In [ ]:
!python -m p24.train_save --data_path "$DATA_PATH" --variant full    --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant compact --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant tiny    --epochs 100 --log_every 10

[P24] train+save | arm=full {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=None aw=- | seed=42 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100%, (4

## 5. Compression evaluation (~20 min, inference only)
fp32 / fp16 / int8-dynamic / prune30-50-70 per variant: params, size, GNN acc,
fused acc (frozen ConvNet, w=0.6), CPU latency.

In [ ]:
!python -m p24.compress_eval --data_path "$DATA_PATH" --convnet_dir "{P21_BASE}"

[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100%, (4, 5):100%, (0, 1):100%, (1, 4):100%
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 18 edges, 90 features each)...
    built 5000/35396 graphs
    built 10000/35396 graphs
    built 15000/35396 graphs
    built 20000/35396 graphs
    built 25000/35396 graphs
    built 3

## 6. Edge-density curve (instant, zero compute)

In [ ]:
!python -m p24.edge_curve --p21_cfg "{P21_CFG}"


  P24 — accuracy vs graph density (P21 Stage A dumps, seed 42)
  full graph = 15 undirected edges/window; * = the locked winner

  config              edges  GNN tgt-test   src-val
  --------------------------------------------------
  top-6  (a=1.0)          6        71.53%    79.95%
  top-9  (a=1.0)          9        71.39%    80.51%
  top-12 (a=1.0)         12        71.69%    79.88%
  top-9  (a=0.5)*         9        74.67%    82.23%

  Reading: edges/window is the per-graph storage knob (vs 15 for a full graph).


## 7. Classic SO — NC + AW retrain arms (~2 h, optional but thesis-core)
Node Coarsening (drop sensor channels → 3/5-node graphs) and Adaptive Windowing
(60→40/30 timesteps). Each arm is one 6-fold run on the `full` architecture.
Resumable; run any subset.

In [ ]:
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc accel_only --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc gyro_only  --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc no_gz      --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --aw 40         --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --aw 30         --epochs 100 --log_every 10

[P24] train+save | arm=full_nc-accel_only {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=accel_only aw=- | seed=42 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_nc-accel_only
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 NC: keeping channels [0, 1, 2] -> 3 nodes/graph
[LODO] P21 edge_mode=pearson  select=topk  topk=3  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (1, 2):100%, (0, 2):100%, (0, 1):100%
    [pearson val] edges/window: mean 3.0  min 3  max 3  | most-kept pairs: (0, 2):100%, (0, 1):100%, (1, 2):100%
    [pearson test] edges/window: mean 3.0  min 3  max 3  | most

## 8. Classic SO — RED / GS edge-budget study (inference only, ~15 min)
Re-evaluates the frozen `full` checkpoints under RED (random edges), GS
(type-priority) and Pearson top-k at matched budgets k ∈ {9, 6, 3} — the direct
"does data-driven selection beat random/heuristic dropping?" table. Requires the
step-4 `full` checkpoints.

In [ ]:
!python -m p24.edge_budget --data_path "$DATA_PATH" --convnet_dir "{P21_BASE}"

[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (3, 5):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (4, 5):100%, (3, 5):100%, (1, 4):100%, (2, 5):100%
    [pearson test] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 5):100%, (1, 2):100%, (4, 5):100%, (0, 1):100%, (1, 4):100%
    feature_mode=hybrid, feat_dim=90, normalize_mode=per_channel
    building 35396 sensor graphs (6 nodes, 18 edges, 90 features each)...
    built 5000/35396 graphs
    built 10000/35396 graphs
    built 15000/35396 graphs
    built 20000/35396 graphs
    built 25000/35396 graphs
    built 3

## 9. NC / AW summary table (instant)
Collects the step-7 arms next to the uncoarsened reference.

In [ ]:
import json, glob, os
import numpy as np
TARGETS = ["KuHar","MotionSense","RealWorld-Thigh","RealWorld-Waist","UCI","WISDM"]
print(f"{'arm':22s} {'src-val':>8s} {'tgt-GNN':>8s}")
for d in sorted(glob.glob("results/checkpoints/*")):
    arm = os.path.basename(d)
    vp = os.path.join(d, "val_summary.json")
    if not os.path.isfile(vp): continue
    vals = json.load(open(vp))
    if len(vals) < 6: print(f"{arm:22s}  (incomplete {len(vals)}/6)"); continue
    accs = []
    for t in TARGETS:
        z = np.load(os.path.join(d, f"gnn_{t}.npz"))
        accs.append((z["probs"].argmax(1)==z["labels"]).mean())
    print(f"{arm:22s} {np.mean(list(vals.values()))*100:7.2f}% {np.mean(accs)*100:7.2f}%")

arm                     src-val  tgt-GNN
compact                  81.45%   69.99%
full                     82.51%   74.90%
full_aw30                82.12%   70.69%
full_aw40                82.34%   71.67%
full_nc-accel_only       75.30%   68.44%
full_nc-gyro_only        78.22%   64.32%
full_nc-no_gz            78.79%   73.65%
tiny                     80.31%   69.87%


---
### Reading it
- The headline thesis artifact is the **storage-vs-accuracy table**: how small can the
  GNN branch get before the fused number degrades? (e.g. "int8 quantization cuts the
  model 4x at ≤0.2pp cost" — whatever the data says.)
- `compact`/`tiny` are single-seed retrains — note that in writing; the post-hoc
  compression rows are deterministic given the checkpoint.
- The edge curve answers the graph-storage question with data already paid for.
- No gate here — P24 is measurement, not a performance lever.

## 3-seed the two headline arms

In [ ]:
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc no_gz --seed 1 --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant full --nc no_gz --seed 2 --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant tiny --seed 1 --epochs 100 --log_every 10
!python -m p24.train_save --data_path "$DATA_PATH" --variant tiny --seed 2 --epochs 100 --log_every 10

[P24] train+save | arm=full_nc-no_gz_s1 {'cnn_embed_dim': 64, 'gnn_hidden': 64, 'cnn_channels': [32, 64]} | nc=no_gz aw=- | seed=1 | targets=['KuHar', 'MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM'] | out=/content/drive/MyDrive/Thesis/SO_Experiments/DAGHAR/P20_GraphRevival_DG/P24_SO_Compression/results/checkpoints/full_nc-no_gz_s1
[LODO] target = KuHar
[LODO] sources = ['MotionSense', 'RealWorld-Thigh', 'RealWorld-Waist', 'UCI', 'WISDM']
[LODO] feature_mode = hybrid  normalize_mode = per_channel
[LODO] samples: train=35396  val=5412  test=144
[LODO] P24 NC: keeping channels [0, 1, 2, 3, 4] -> 5 nodes/graph
[LODO] P21 edge_mode=pearson  select=topk  topk=9  tau=0.3  alpha=0.5
    [pearson train] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (3, 4):100%, (1, 2):100%, (1, 4):100%, (0, 2):100%, (0, 1):100%
    [pearson val] edges/window: mean 9.0  min 9  max 9  | most-kept pairs: (0, 2):100%, (1, 4):100%, (0, 1):100%, (3, 4):100%, (0, 3):100%
    [pearson te

## Seed-averaged headline summary

In [ ]:
import os
P21_SWEEP = os.path.dirname(P21_BASE)   # .../P21_PearsonGraph/results/sweep
!python -m p24.seed_summary --p21_sweep "{P21_SWEEP}"


  P24 — SEED-AVERAGED headline SO arms (fused w=0.6, frozen ConvNet/seed)
  Reference: full system 78.10 ± 0.14% (P21, seeds 42/1/2)

  arm                  seeds  fused mean±std  GNN-only  Δ vs full
  ----------------------------------------------------------------
  full (P21 winner)        3    78.10±0.14%    75.19% (reference)
  full_nc-no_gz            3    78.15±0.83%    72.73%    +0.05pp
  tiny                     3    76.94±0.31%    71.66%    -1.16pp

  Per-fold fused seed-means:
  fold              full (P21 winn   full_nc-no_gz            tiny
  KuHar                     68.29%          65.97%          64.58%
  MotionSense               87.88%          87.85%          85.88%
  RealWorld-Thigh           75.02%          75.07%          74.83%
  RealWorld-Waist           75.67%          76.38%          75.99%
  UCI                       85.85%          84.59%          85.89%
  WISDM                     75.92%          79.05%          74.49%

  Single-seed arms (compact, accel_o